# Data trigger stitching

Extract multiplicative weights that place the MinimumBias, Jet60, Jet80, and Jet100 reconstructed-jet spectra on one common MinimumBias yield scale. The default observable is the inclusive-jet $p_T$ spectrum in $-1.6 \leq \eta_{Lab} < 1.6$; configurable CM-frame and dijet-$p_T^{ave}$ alternatives use the same workflow.

Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

<!-- detailed-workflow-guide -->

### Detailed workflow and data interpretation

For each trigger, the selected two-dimensional histogram is projected onto its momentum axis over the half-open interval $[\eta_{min},\eta_{max})$. No unit-area or bin-width normalization is applied: the stored weighted yields and their `Sumw2` uncertainties are retained.

At each adjacent trigger boundary, the local scale is the ratio of the reference and candidate integrals in the configured overlap, $s=B/A$. Its statistical uncertainty assumes independent trigger samples, $(\sigma_s/s)^2=(\sigma_B/B)^2+(\sigma_A/A)^2$. Cumulative weights multiply the adjacent factors, and their reported uncertainties are propagated as independent. The global normalization uncertainty is reported with each weight but is not folded into every plotted bin error; plotted bin errors are the stored statistical errors multiplied by the central weight.

The sequence is MB $\leftarrow$ Jet60 in 90--140 GeV, scaled Jet60 $\leftarrow$ Jet80 in 110--180 GeV, and scaled Jet80 $\leftarrow$ Jet100 in 130--200 GeV. Ratios use ROOT's standard independent-error propagation, never the binomial option. A flat ratio near unity across an overlap supports a single normalization factor; visible slopes or thresholds signal that the overlap or trigger choice needs review.

## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.

In [9]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from dataclasses import asdict, replace
from pathlib import Path
import csv
import json
import os
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'CMakeLists.txt').is_file()
     and (candidate / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate jetAnalysis; start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.data_trigger_stitching import (
    derive_scale, draw_comparison, draw_final_overlay,
    load_pt_projection, scaled_clone,
)
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.plot_labels import CMS_PRELIMINARY_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.root_style import DEFAULT_PLOT_STYLE, configure_cms_header
CMS_HEADER_LEFT = CMS_PRELIMINARY_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = False  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# Cell role: configure deterministic ROOT drawing behavior.
# Interpretation: These settings affect presentation only, not yields or fitted weights.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

Set `OBSERVABLE = 'inclusive'` (default) or `'dijet'`, and choose `FRAME = 'lab'` or `'cm'`. For dijets, `DIJET_ETA_CUT_INDEX` selects the production-level single-jet acceptance from `ETA_CUTS`; it is distinct from `PROJECTION_ETA_RANGE`, which selects the displayed dijet-axis interval before projecting $p_T^{ave}$.

In [11]:
ETA_CUT_VALUE = 1.9  # per-jet |etaCM| <= value; supported producer cuts only
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define and validate user-facing stitching inputs and output controls.
# Interpretation: The observable, frame, eta selections, and overlap windows directly affect the weights.
# The preceding Markdown gives the equations and physics assumptions for this step.
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
DATA_DIRECTION = 'combined'  # combined, Pbgoing, or pgoing
DATA_SELECTION = 'jetId'     # jetId, trkMax, or noSel
OBSERVABLE = 'dijet'     # inclusive or dijet
FRAME = 'cm'                # lab or cm
PROJECTION_ETA_RANGE = (-1.9, 1.9)
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUTS = DIJET_ETA_CUTS  # shared current production cut order
DIJET_ETA_CUT_INDEX = eta_cut_index(ETA_CUT_VALUE)  # 1.9
MATCHING_RANGES = {
    'Jet60': (90.0, 140.0),
    'Jet80': (110.0, 180.0),
    'Jet100': (130.0, 250.0),
}
PT_DISPLAY_RANGE = (40.0, 360.0)
PAIR_RATIO_RANGE = (0.1, 1.3)
FINAL_RATIO_RANGE = (0.1, 1.3)
REBIN_PT = 1
LOG_Y = True
DRAW_GRID = True
SAVE_PNG = False
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
OUTPUT_DIR = Path(os.environ.get(
    'DATA_TRIGGER_STITCHING_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/data_trigger_stitching',
)) / DATA_DIRECTION / DATA_SELECTION / f'{OBSERVABLE}_{FRAME}'

def filename_number(value):
    return f'{value:g}'.replace('-', 'm').replace('.', 'p')

eta_range_tag = (
    f'eta{filename_number(PROJECTION_ETA_RANGE[0])}'
    f'to{filename_number(PROJECTION_ETA_RANGE[1])}'
)
dijet_cut_tag = (
    f'_jeteta{filename_number(ETA_CUTS[DIJET_ETA_CUT_INDEX])}'
    if OBSERVABLE == 'dijet' else ''
)
matching_tag = '_'.join(
    f'{trigger.lower()}pt{filename_number(low)}to{filename_number(high)}'
    for trigger, (low, high) in MATCHING_RANGES.items()
)
STITCHING_TAG = (
    f'{DATA_DIRECTION}_{DATA_SELECTION}_{OBSERVABLE}_{FRAME}_'
    f'{eta_range_tag}{dijet_cut_tag}_{matching_tag}'
)
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE, annotation_text_size=0.028, legend_text_size=0.030,
)

if DATA_DIRECTION not in {'combined', 'Pbgoing', 'pgoing'}:
    raise ValueError(f'Unsupported DATA_DIRECTION={DATA_DIRECTION!r}')
if DATA_SELECTION not in {'jetId', 'trkMax', 'noSel'}:
    raise ValueError(f'Unsupported DATA_SELECTION={DATA_SELECTION!r}')
if OBSERVABLE not in {'inclusive', 'dijet'}:
    raise ValueError(f'Unsupported OBSERVABLE={OBSERVABLE!r}')
if FRAME not in {'lab', 'cm'}:
    raise ValueError(f'Unsupported FRAME={FRAME!r}')
if not 0 <= DIJET_ETA_CUT_INDEX < len(ETA_CUTS):
    raise ValueError('DIJET_ETA_CUT_INDEX is outside ETA_CUTS')
if isinstance(REBIN_PT, bool) or not isinstance(REBIN_PT, int) or REBIN_PT < 1:
    raise ValueError('REBIN_PT must be a positive integer')
DATA_FILES = {
    trigger: resolve_data_file(DATA_DIR, trigger, DATA_DIRECTION, DATA_SELECTION)
    for trigger in TRIGGERS
}
missing = [str(path) for path in DATA_FILES.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing ROOT files:\n' + '\n'.join(missing))
DATA_FILES

{'MinimumBias': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/mb_ak4_jetId.root'),
 'Jet60': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/jet60_ak4_jetId.root'),
 'Jet80': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/jet80_ak4_jetId.root'),
 'Jet100': PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/exp/jet100_ak4_jetId.root')}

## Load and project all trigger spectra

In [12]:
# Cell role: load each configured TH2 and project momentum over the selected eta range.
# Interpretation: These are unnormalized stored yields; trigger prescales and exposure remain visible.
# The preceding Markdown gives the equations and physics assumptions for this step.
raw_spectra = {}
resolved_keys = {}
for trigger in TRIGGERS:
    spectrum, key = load_pt_projection(
        DATA_FILES[trigger], observable=OBSERVABLE, frame=FRAME,
        eta_range=PROJECTION_ETA_RANGE, eta_cut_index=DIJET_ETA_CUT_INDEX,
        name=f'hPt_{trigger}_{OBSERVABLE}_{FRAME}',
    )
    if REBIN_PT > 1:
        spectrum.Rebin(REBIN_PT)
    raw_spectra[trigger] = spectrum
    resolved_keys[trigger] = key
resolved_keys

{'MinimumBias': 'hRecoDijetPtEtaCM_5',
 'Jet60': 'hRecoDijetPtEtaCM_5',
 'Jet80': 'hRecoDijetPtEtaCM_5',
 'Jet100': 'hRecoDijetPtEtaCM_5'}

## MB and Jet60

First compare the raw Jet60 spectrum to MB, extract the 90--140 GeV integral-ratio weight, then repeat the overlay after applying that factor.

In [13]:
# Cell role: extract the Jet60-to-MB normalization and validate it visually.
# Interpretation: Independent trigger yields determine one constant factor in the configured overlap.
# The preceding Markdown gives the equations and physics assumptions for this step.
eta_symbol = '#eta_{CM}' if FRAME == 'cm' else '#eta_{lab}'
eta_label = f'{PROJECTION_ETA_RANGE[0]:g} <= {eta_symbol} < {PROJECTION_ETA_RANGE[1]:g}'
common_annotation = [f'{OBSERVABLE.capitalize()} jets, {FRAME.upper()} frame', eta_label]
mb_jet60_raw = draw_comparison(
    raw_spectra['MinimumBias'], raw_spectra['Jet60'],
    reference_label='MB', candidate_label='Jet60',
    annotation_lines=common_annotation,
    output_path=OUTPUT_DIR / f'01_mb_jet60_raw_{STITCHING_TAG}.pdf',
    display_range=PT_DISPLAY_RANGE, ratio_range=PAIR_RATIO_RANGE,
    log_y=LOG_Y, save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)
jet60_scale = derive_scale(
    raw_spectra['MinimumBias'], raw_spectra['Jet60'],
    reference_label='MB', candidate_label='Jet60',
    pt_range=MATCHING_RANGES['Jet60'],
)
scaled_spectra = {'MinimumBias': raw_spectra['MinimumBias']}
scaled_spectra['Jet60'] = scaled_clone(
    raw_spectra['Jet60'], jet60_scale.cumulative_scale, 'hPt_Jet60_scaledToMB',
)
mb_jet60_scaled = draw_comparison(
    scaled_spectra['MinimumBias'], scaled_spectra['Jet60'],
    reference_label='MB', candidate_label='Jet60 x weight',
    annotation_lines=[*common_annotation,
                      f'weight = {jet60_scale.cumulative_scale:.6g} +/- {jet60_scale.cumulative_uncertainty:.2g}'],
    output_path=OUTPUT_DIR / f'02_mb_jet60_scaled_{STITCHING_TAG}.pdf',
    display_range=PT_DISPLAY_RANGE, ratio_range=PAIR_RATIO_RANGE,
    log_y=LOG_Y, save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)
display(mb_jet60_raw['canvas']); display(mb_jet60_scaled['canvas'])
jet60_scale

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/01_mb_jet60_raw_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/02_mb_jet60_scaled_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.pdf has been created


StitchingScale(reference='MB', candidate='Jet60', pt_range=(90.0, 140.0), local_scale=0.20446067562279902, local_uncertainty=0.0004706569495243109, cumulative_scale=0.20446067562279902, cumulative_uncertainty=0.0004706569495243109, reference_integral=227302.0, candidate_integral=1111715.0)

## Scaled Jet60 and Jet80

Use the already MB-normalized Jet60 spectrum as the reference in 110--180 GeV. The resulting Jet80 cumulative weight therefore places Jet80 directly on the MB scale.

In [14]:
# Cell role: extract the Jet80 cumulative normalization through scaled Jet60.
# Interpretation: The local factor matches Jet80 to scaled Jet60; cumulative propagation retains the MB scale.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet60_jet80_raw = draw_comparison(
    scaled_spectra['Jet60'], raw_spectra['Jet80'],
    reference_label='Jet60 x MB weight', candidate_label='Jet80',
    annotation_lines=common_annotation,
    output_path=OUTPUT_DIR / f'03_jet60_jet80_raw_{STITCHING_TAG}.pdf',
    display_range=PT_DISPLAY_RANGE, ratio_range=PAIR_RATIO_RANGE,
    log_y=LOG_Y, save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)
jet80_scale = derive_scale(
    raw_spectra['Jet60'], raw_spectra['Jet80'],
    reference_label='Jet60', candidate_label='Jet80',
    pt_range=MATCHING_RANGES['Jet80'],
    reference_cumulative_scale=jet60_scale.cumulative_scale,
    reference_cumulative_uncertainty=jet60_scale.cumulative_uncertainty,
)
scaled_spectra['Jet80'] = scaled_clone(
    raw_spectra['Jet80'], jet80_scale.cumulative_scale, 'hPt_Jet80_scaledToMB',
)
jet60_jet80_scaled = draw_comparison(
    scaled_spectra['Jet60'], scaled_spectra['Jet80'],
    reference_label='Jet60 x MB weight', candidate_label='Jet80 x MB weight',
    annotation_lines=[*common_annotation,
                      f'Jet80 MB weight = {jet80_scale.cumulative_scale:.6g} +/- {jet80_scale.cumulative_uncertainty:.2g}'],
    output_path=OUTPUT_DIR / f'04_jet60_jet80_scaled_{STITCHING_TAG}.pdf',
    display_range=PT_DISPLAY_RANGE, ratio_range=PAIR_RATIO_RANGE,
    log_y=LOG_Y, save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)
display(jet60_jet80_raw['canvas']); display(jet60_jet80_scaled['canvas'])
jet80_scale

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/03_jet60_jet80_raw_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_hPt_Jet80_scaledToMB_over_hPt_Jet60_scaledToMB
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/04_jet60_jet80_scaled_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.pdf has been created


StitchingScale(reference='Jet60', candidate='Jet80', pt_range=(110.0, 180.0), local_scale=0.12407853672599503, local_uncertainty=0.00018366363977318057, cumulative_scale=0.025369181449285225, cumulative_uncertainty=6.943002380817635e-05, reference_integral=513031.0, candidate_integral=4134728.0)

## Scaled Jet80 and Jet100

Use the already MB-normalized Jet80 spectrum as the reference in 130--200 GeV, producing the final Jet100-to-MB cumulative weight.

In [15]:
# Cell role: extract the Jet100 cumulative normalization through scaled Jet80.
# Interpretation: The final factor closes the adjacent-trigger chain onto the MB yield scale.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet80_jet100_raw = draw_comparison(
    scaled_spectra['Jet80'], raw_spectra['Jet100'],
    reference_label='Jet80 x MB weight', candidate_label='Jet100',
    annotation_lines=common_annotation,
    output_path=OUTPUT_DIR / f'05_jet80_jet100_raw_{STITCHING_TAG}.pdf',
    display_range=PT_DISPLAY_RANGE, ratio_range=PAIR_RATIO_RANGE,
    log_y=LOG_Y, save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)
jet100_scale = derive_scale(
    raw_spectra['Jet80'], raw_spectra['Jet100'],
    reference_label='Jet80', candidate_label='Jet100',
    pt_range=MATCHING_RANGES['Jet100'],
    reference_cumulative_scale=jet80_scale.cumulative_scale,
    reference_cumulative_uncertainty=jet80_scale.cumulative_uncertainty,
)
scaled_spectra['Jet100'] = scaled_clone(
    raw_spectra['Jet100'], jet100_scale.cumulative_scale, 'hPt_Jet100_scaledToMB',
)
jet80_jet100_scaled = draw_comparison(
    scaled_spectra['Jet80'], scaled_spectra['Jet100'],
    reference_label='Jet80 x MB weight', candidate_label='Jet100 x MB weight',
    annotation_lines=[*common_annotation,
                      f'Jet100 MB weight = {jet100_scale.cumulative_scale:.6g} +/- {jet100_scale.cumulative_uncertainty:.2g}'],
    output_path=OUTPUT_DIR / f'06_jet80_jet100_scaled_{STITCHING_TAG}.pdf',
    display_range=PT_DISPLAY_RANGE, ratio_range=PAIR_RATIO_RANGE,
    log_y=LOG_Y, save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)
display(jet80_jet100_raw['canvas']); display(jet80_jet100_scaled['canvas'])
jet100_scale

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/05_jet80_jet100_raw_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_hPt_Jet100_scaledToMB_over_hPt_Jet80_scaledToMB
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/06_jet80_jet100_scaled_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.pdf has been created


StitchingScale(reference='Jet80', candidate='Jet100', pt_range=(130.0, 250.0), local_scale=0.9202967166436355, local_uncertainty=0.0008597283026111665, cumulative_scale=0.02334717439171382, cumulative_uncertainty=6.751614416926796e-05, reference_integral=2200400.0, candidate_integral=2390968.0)

## Final MB-scale overlay, ratios, and exported weights

Overlay the four spectra on the common MB scale and show Jet60/MB, Jet80/MB, and Jet100/MB. Export both local adjacent-trigger factors and cumulative MB-scale weights as JSON and CSV for downstream analysis.

In [16]:
# Cell role: summarize the stitched spectra and persist the extracted normalization parameters.
# Interpretation: Final ratios test global consistency; exported cumulative factors multiply raw trigger yields.
# The preceding Markdown gives the equations and physics assumptions for this step.
scales = [jet60_scale, jet80_scale, jet100_scale]
final_result = draw_final_overlay(
    scaled_spectra, annotation_lines=common_annotation,
    output_path=OUTPUT_DIR / f'07_all_triggers_scaled_to_mb_{STITCHING_TAG}.pdf',
    display_range=PT_DISPLAY_RANGE, ratio_range=FINAL_RATIO_RANGE,
    log_y=LOG_Y, save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
scale_records = [asdict(scale) for scale in scales]
def histogram_key_for_trigger(label):
    trigger = label.split()[0]
    trigger = {'MB': 'MinimumBias'}.get(trigger, trigger)
    return resolved_keys[trigger]

rows = [{
    'data_direction': DATA_DIRECTION,
    'data_selection': DATA_SELECTION,
    'observable': OBSERVABLE,
    'frame': FRAME,
    'projection_eta_low': PROJECTION_ETA_RANGE[0],
    'projection_eta_high': PROJECTION_ETA_RANGE[1],
    'dijet_eta_cut_index': DIJET_ETA_CUT_INDEX if OBSERVABLE == 'dijet' else '',
    'dijet_jet_eta_cut': ETA_CUTS[DIJET_ETA_CUT_INDEX] if OBSERVABLE == 'dijet' else '',
    'reference_histogram_key': histogram_key_for_trigger(scale.reference),
    'candidate_histogram_key': histogram_key_for_trigger(scale.candidate),
    'reference_trigger': scale.reference,
    'candidate_trigger': scale.candidate,
    'matching_pt_low': scale.pt_range[0],
    'matching_pt_high': scale.pt_range[1],
    'local_scale': scale.local_scale,
    'local_uncertainty': scale.local_uncertainty,
    'cumulative_scale': scale.cumulative_scale,
    'cumulative_uncertainty': scale.cumulative_uncertainty,
    'reference_integral': scale.reference_integral,
    'candidate_integral': scale.candidate_integral,
} for scale in scales]
metadata = {
    'observable': OBSERVABLE, 'frame': FRAME,
    'projection_eta_range': PROJECTION_ETA_RANGE,
    'dijet_eta_cut_index': DIJET_ETA_CUT_INDEX if OBSERVABLE == 'dijet' else None,
    'dijet_eta_cut': ETA_CUTS[DIJET_ETA_CUT_INDEX] if OBSERVABLE == 'dijet' else None,
    'direction': DATA_DIRECTION, 'selection': DATA_SELECTION,
    'stitching_tag': STITCHING_TAG,
    'matching_ranges': MATCHING_RANGES,
    'histogram_keys': resolved_keys, 'scales': scale_records,
}
json_path = OUTPUT_DIR / f'trigger_stitching_weights_{STITCHING_TAG}.json'
json_path.write_text(json.dumps(metadata, indent=2) + '\n')
csv_path = OUTPUT_DIR / f'trigger_stitching_weights_{STITCHING_TAG}.csv'
with csv_path.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)
print('Cumulative trigger weights on the MB scale:')
print('  MinimumBias: 1 (definition)')
for scale in scales:
    print(f'  {scale.candidate}: {scale.cumulative_scale:.8g} +/- {scale.cumulative_uncertainty:.3g}')
print(f'Wrote {json_path}')
print(f'Wrote {csv_path}')
display(final_result['canvas'])

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_data_trigger_stitching_final


Final-overlay global maximum: trigger=Jet60, value=539995.98, x_bin=6, x_center=55 GeV, x_edges=[50, 60) GeV
Cumulative trigger weights on the MB scale:
  MinimumBias: 1 (definition)
  Jet60: 0.20446068 +/- 0.000471
  Jet80: 0.025369181 +/- 6.94e-05
  Jet100: 0.023347174 +/- 6.75e-05
Wrote /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/trigger_stitching_weights_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.json
Wrote /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/trigger_stitching_weights_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.csv


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/data_trigger_stitching/combined/jetId/dijet_cm/07_all_triggers_scaled_to_mb_combined_jetId_dijet_cm_etam1p9to1p9_jeteta1p9_jet60pt90to140_jet80pt110to180_jet100pt130to250.pdf has been created
